# PP-StructureV3 в Google Colab: тест на страницах аварской газеты

Эта тетрадка подготовлена для **Google Colab**. В ней проверяем, как **PaddleOCR PP-StructureV3** работает на отрендеренных PNG-страницах из `train_pdfs/img/`.

## Что проверяем

1. **Layout detection** — детекция областей страницы (текст, заголовки, картинки, таблицы и т.д.)
2. **Блоки, колонки, заголовки** — какие типы областей находит модель на газетной вёрстке
3. **Порядок чтения** — в каком порядке блоки попадают в итоговый результат
4. **Экспорт** — сохранение в JSON и Markdown

## Как подготовить данные для Colab

Нужно, чтобы PNG лежали так:

```text
/content/train_pdfs/img/
  page_001.png
  page_002.png
```

Ниже есть ячейка, которая позволяет загрузить:

- отдельные `.png` файлы;
- `.zip` архив с PNG;
- или подключить Google Drive и взять данные оттуда.

## Документация

- [PP-StructureV3 tutorial](https://paddlepaddle.github.io/PaddleOCR/main/en/version3.x/pipeline_usage/PP-StructureV3.html)
- [Установка PaddleOCR 3.x](https://paddlepaddle.github.io/PaddleOCR/main/en/version3.x/installation.html)

> **Важно:** при первом запуске скачаются модели (несколько ГБ). На CPU inference может занять несколько минут на одну страницу.

## 0. Установка зависимостей в Colab

Выполните ячейку ниже **один раз** после запуска runtime.

- `paddlepaddle` — фреймворк
- `paddleocr[doc-parser]` — группа зависимостей для PP-StructureV3
- `matplotlib`, `pandas`, `pillow` — визуализация и таблицы

После установки Colab может попросить **Restart runtime**. Если попросит — перезапустите runtime и продолжайте со следующей ячейки.

Для более быстрого inference можно включить GPU: `Runtime` → `Change runtime type` → `T4 GPU`. Даже с GPU первый запуск будет долгим из-за скачивания моделей.

In [ ]:
# Colab: установка зависимостей
# Если Colab попросит перезапустить runtime после установки, перезапустите и продолжайте ниже.

%pip install -q -U "paddlepaddle>=3.0.0" "paddleocr[doc-parser]" matplotlib pandas pillow

## 1. Загрузка данных в Colab

Целевая папка для входных страниц:

```text
/content/train_pdfs/img/
```

Варианты:

1. Загрузить `.png` файлы напрямую через кнопку загрузки Colab.
2. Загрузить `.zip` архив с PNG — он будет распакован в `/content/train_pdfs/img/`.
3. Подключить Google Drive и скопировать оттуда папку или архив.

Если вы уже заранее положили файлы в `/content/train_pdfs/img/`, ячейку загрузки можно пропустить.

In [ ]:
from pathlib import Path
import shutil
import zipfile

PROJECT_ROOT = Path("/content")
IMAGE_DIR = PROJECT_ROOT / "train_pdfs" / "img"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "pp_structure_v3"
IMAGE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Папка для входных PNG: {IMAGE_DIR}")
print(f"Папка для результатов: {OUTPUT_DIR}")

In [ ]:
# Вариант A: загрузить PNG или ZIP прямо с компьютера в Colab.
# Запустите эту ячейку, если файлы еще не лежат в /content/train_pdfs/img/.

from google.colab import files

uploaded = files.upload()

for filename in uploaded:
    src = Path(filename)
    if src.suffix.lower() == ".zip":
        with zipfile.ZipFile(src) as zf:
            zf.extractall(IMAGE_DIR)
        src.unlink()
        print(f"Распакован архив: {filename}")
    elif src.suffix.lower() == ".png":
        dst = IMAGE_DIR / src.name
        shutil.move(str(src), dst)
        print(f"Загружен PNG: {dst.name}")
    else:
        print(f"Пропущен файл не PNG/ZIP: {filename}")

print(f"PNG в {IMAGE_DIR}: {len(list(IMAGE_DIR.rglob('*.png')))}")

In [ ]:
# Вариант B: подключить Google Drive и скопировать данные оттуда.
# Раскомментируйте и поправьте DRIVE_SOURCE, если данные лежат на Drive.

# from google.colab import drive
# drive.mount('/content/drive')

# DRIVE_SOURCE = Path('/content/drive/MyDrive/ann_avar_ocr_finetuning/train_pdfs/img')
# if DRIVE_SOURCE.is_dir():
#     for png_path in DRIVE_SOURCE.rglob('*.png'):
#         shutil.copy2(png_path, IMAGE_DIR / png_path.name)
#     print(f'Скопировано PNG: {len(list(IMAGE_DIR.glob("*.png")))}')
# else:
#     raise FileNotFoundError(f'Не найдена папка: {DRIVE_SOURCE}')

## 2. Выбор тестовой страницы

По умолчанию берём первую PNG из `/content/train_pdfs/img/`.  
Чтобы выбрать другую страницу, поменяйте `IMAGE_INDEX` или задайте `IMAGE_PATH` вручную.

In [ ]:
images = sorted(IMAGE_DIR.rglob("*.png"))
if not images:
    raise FileNotFoundError(
        f"Нет PNG в {IMAGE_DIR}. Загрузите PNG/ZIP в ячейке выше или скопируйте файлы с Google Drive."
    )

IMAGE_INDEX = 0  # поменяйте индекс, если нужна другая страница
IMAGE_PATH = images[IMAGE_INDEX]

print(f"Тестовая страница: {IMAGE_PATH.name}")
print(f"Всего PNG найдено: {len(images)}")
print(f"Результаты будут в: {OUTPUT_DIR}")

In [ ]:
from IPython.display import Image, display

display(Image(filename=str(IMAGE_PATH), width=700))

## 3. Инициализация пайплайна PP-StructureV3

Ключевые модули для газетной вёрстки:

| Параметр | Зачем |
|----------|-------|
| `use_region_detection=True` | **Region detection** — отделяет статьи/колонки на одной странице (важно для газет) |
| `use_doc_orientation_classify` | Поворот страницы (0°/90°/180°/270°) |
| `use_doc_unwarping` | Выпрямление искажённого скана |
| `use_table_recognition` | Распознавание таблиц |
| `use_formula_recognition` | Формулы (для газет обычно не нужно, можно отключить для скорости) |
| `device="gpu"` | GPU-инференс, если Runtime Colab запущен с GPU |

Модель layout detection по умолчанию — **PP-DocLayout_plus** (23 класса: `text`, `paragraph_title`, `doc_title`, `image`, `table`, `header`, `footer` и др.).

Если в Colab включён GPU, поставьте `DEVICE = "gpu"`. Если возникают ошибки с GPU-сборкой PaddlePaddle, оставьте `DEVICE = None` — пайплайн пойдёт на CPU.

In [ ]:
from paddleocr import PPStructureV3

# Для Colab CPU оставьте None.
# Если включили Runtime -> Change runtime type -> T4 GPU, можно попробовать DEVICE = "gpu".
DEVICE = None

pipeline_kwargs = dict(
    # Для чистых PNG из PDF обычно не нужны препроцессоры:
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    # Region detection помогает на многоколоночных газетных страницах:
    use_region_detection=True,
    # Ускорение: отключаем то, что в газете редко встречается
    use_formula_recognition=False,
    use_seal_recognition=False,
)

if DEVICE:
    pipeline_kwargs["device"] = DEVICE

pipeline = PPStructureV3(**pipeline_kwargs)

print("Пайплайн инициализирован.")
print("Device:", DEVICE or "cpu/default")

## 4. Запуск inference

`predict()` возвращает список объектов результата — по одному на каждую страницу (для PNG это один элемент).

In [ ]:
results = pipeline.predict(
    input=str(IMAGE_PATH),
    # format_block_content=True  # форматировать block_content как Markdown
)

result = results[0]  # одна страница
print(f"Получено результатов: {len(results)}")

## 5. Layout detection

Сырые боксы layout-модели лежат в `layout_det_res.boxes`:
- `label` — тип области (`text`, `paragraph_title`, `image`, ...)
- `coordinate` — `[x1, y1, x2, y2]`
- `score` — уверенность детектора

In [ ]:
import json
import pandas as pd

data = result.json["res"]
layout_boxes = data.get("layout_det_res", {}).get("boxes", [])

layout_df = pd.DataFrame([
    {
        "label": box["label"],
        "score": round(box["score"], 3),
        "x1": round(box["coordinate"][0]),
        "y1": round(box["coordinate"][1]),
        "x2": round(box["coordinate"][2]),
        "y2": round(box["coordinate"][3]),
    }
    for box in layout_boxes
])

print(f"Найдено layout-областей: {len(layout_df)}")
print("\nРаспределение по типам:")
display(layout_df["label"].value_counts().to_frame("count"))
layout_df.head(10)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image as PILImage

# Цвета для основных типов блоков
LABEL_COLORS = {
    "text": "#4C78A8",
    "paragraph_title": "#F58518",
    "doc_title": "#E45756",
    "image": "#72B7B2",
    "figure": "#72B7B2",
    "table": "#54A24B",
    "header": "#B279A2",
    "footer": "#9D755D",
}
DEFAULT_COLOR = "#BAB0AC"

img = PILImage.open(IMAGE_PATH).convert("RGB")
fig, ax = plt.subplots(figsize=(12, 16))
ax.imshow(img)
ax.axis("off")
ax.set_title(f"Layout detection: {IMAGE_PATH.name}")

for box in layout_boxes:
    x1, y1, x2, y2 = box["coordinate"]
    label = box["label"]
    color = LABEL_COLORS.get(label, DEFAULT_COLOR)
    rect = patches.Rectangle(
        (x1, y1), x2 - x1, y2 - y1,
        linewidth=1.5, edgecolor=color, facecolor="none",
    )
    ax.add_patch(rect)
    ax.text(x1, max(y1 - 2, 0), label, fontsize=7, color=color,
            bbox=dict(facecolor="white", alpha=0.6, edgecolor="none", pad=1))

plt.tight_layout()
plt.show()

## 6. Блоки, колонки и заголовки

После layout detection и OCR пайплайн формирует **`parsing_res_list`** — список блоков в **порядке чтения**.

Поля каждого блока:
- `block_label` — тип (`text`, `paragraph_title`, `image`, ...)
- `block_bbox` — координаты
- `block_content` — распознанный текст (или описание для картинок/таблиц)
- `block_order` — порядковый номер в reading order (`None`, если порядок не определён)
- `block_id` — индекс блока

Для газет обратите внимание:
- сколько блоков помечено как `paragraph_title` / `doc_title`;
- как группируются текстовые колонки слева направо;
- не путает ли модель колонки разных статей.

In [ ]:
blocks = data.get("parsing_res_list", [])
print(f"Блоков после парсинга: {len(blocks)}")

blocks_df = pd.DataFrame([
    {
        "block_order": block.get("block_order"),
        "block_id": block.get("block_id"),
        "block_label": block.get("block_label"),
        "content_preview": (block.get("block_content") or "")[:120].replace("\n", " "),
    }
    for block in blocks
])

display(blocks_df)

In [ ]:
# Отдельно смотрим заголовки
title_labels = {"doc_title", "paragraph_title", "figure_title", "table_title"}
titles = blocks_df[blocks_df["block_label"].isin(title_labels)]

print(f"Заголовков найдено: {len(titles)}")
titles

## 7. Восстановление порядка чтения

Официально: **порядок элементов в `parsing_res_list` = reading order** после постобработки.

Дополнительно можно сравнить с `block_order` и визуализацией из `save_to_img()`.

In [ ]:
reading_order_df = blocks_df.copy()
reading_order_df["list_index"] = range(len(blocks_df))

# Сортируем по block_order, если он задан
ordered = reading_order_df.sort_values(
    by=["block_order", "list_index"],
    na_position="last",
)

print("Порядок чтения (block_order → list_index):")
display(ordered[["block_order", "list_index", "block_label", "content_preview"]])

In [ ]:
# Визуализация reading order: номер блока на bbox
img = PILImage.open(IMAGE_PATH).convert("RGB")
fig, ax = plt.subplots(figsize=(12, 16))
ax.imshow(img)
ax.axis("off")
ax.set_title("Reading order (номер = позиция в parsing_res_list)")

for idx, block in enumerate(blocks):
    bbox = block.get("block_bbox")
    if bbox is None:
        continue
    x1, y1, x2, y2 = bbox
    rect = patches.Rectangle(
        (x1, y1), x2 - x1, y2 - y1,
        linewidth=1.5, edgecolor="red", facecolor="none",
    )
    ax.add_patch(rect)
    order = block.get("block_order")
    label = f"{idx}" if order is None else f"{idx}|o{order}"
    ax.text(x1 + 4, y1 + 14, label, fontsize=9, color="red",
            bbox=dict(facecolor="yellow", alpha=0.7, edgecolor="none", pad=1))

plt.tight_layout()
plt.show()

## 8. Встроенная визуализация PaddleOCR

`result.save_to_img()` сохраняет картинки с разметкой layout, OCR, таблиц и т.д.

In [ ]:
vis_dir = OUTPUT_DIR / IMAGE_PATH.stem / "vis"
vis_dir.mkdir(parents=True, exist_ok=True)

result.save_to_img(save_path=str(vis_dir))

vis_images = sorted(vis_dir.glob("*.png"))
print(f"Сохранено визуализаций: {len(vis_images)}")
for path in vis_images:
    print(" -", path.name)
    display(Image(filename=str(path), width=700))

## 9. Экспорт в JSON

`save_to_json()` сохраняет полный структурированный результат: layout, OCR, parsing_res_list и настройки моделей.

In [ ]:
json_dir = OUTPUT_DIR / IMAGE_PATH.stem / "json"
json_dir.mkdir(parents=True, exist_ok=True)

result.save_to_json(save_path=str(json_dir), indent=2, ensure_ascii=False)

json_files = sorted(json_dir.glob("*.json"))
print("JSON файлы:")
for path in json_files:
    print(" -", path)

# Краткий просмотр ключей верхнего уровня
if json_files:
    with open(json_files[0], encoding="utf-8") as f:
        saved = json.load(f)
    print("\nКлючи в res:", list(saved.get("res", {}).keys()))

## 10. Экспорт в Markdown

Два способа:
1. `save_to_markdown()` — готовый `.md` на страницу
2. `result.markdown` + `concatenate_markdown_pages()` — для многостраничных PDF

In [ ]:
md_dir = OUTPUT_DIR / IMAGE_PATH.stem / "markdown"
md_dir.mkdir(parents=True, exist_ok=True)

result.save_to_markdown(save_path=str(md_dir))

md_files = sorted(md_dir.glob("*.md"))
print("Markdown файлы:")
for path in md_files:
    print(" -", path)
    print(path.read_text(encoding="utf-8")[:1500])
    print("\n---\n")

In [ ]:
# Альтернатива: получить markdown прямо из объекта результата
md_info = result.markdown
print("Ключи markdown:", md_info.keys())
print("\nПревью текста:")
print((md_info.get("markdown_texts") or "")[:1500])

## 11. (Опционально) Сравнить с/без region detection

На газетных полосах region detection должен лучше отделять статьи.  
Запустите ячейку ниже и сравните количество блоков и порядок чтения.

In [ ]:
def run_and_summarize(use_region_detection: bool) -> pd.DataFrame:
    kwargs = dict(pipeline_kwargs)
    kwargs["use_region_detection"] = use_region_detection

    pipe = PPStructureV3(**kwargs)
    res = pipe.predict(input=str(IMAGE_PATH))[0]
    blocks = res.json["res"].get("parsing_res_list", [])
    return pd.DataFrame([
        {
            "mode": "with_region" if use_region_detection else "no_region",
            "block_order": b.get("block_order"),
            "block_label": b.get("block_label"),
            "preview": (b.get("block_content") or "")[:80],
        }
        for b in blocks
    ])

# Внимание: второй прогон снова грузит модели и может быть долгим
# compare_df = pd.concat([
#     run_and_summarize(True),
#     run_and_summarize(False),
# ], ignore_index=True)
# compare_df

## 12. Скачать результаты из Colab

Результаты лежат в `/content/outputs/pp_structure_v3/`.  
Ячейка ниже упакует их в ZIP и предложит скачать на компьютер.

In [ ]:
from google.colab import files

zip_path = PROJECT_ROOT / "pp_structure_v3_outputs.zip"
if zip_path.exists():
    zip_path.unlink()

shutil.make_archive(str(zip_path.with_suffix("")), "zip", OUTPUT_DIR)
print(f"Архив готов: {zip_path}")
files.download(str(zip_path))

## 13. Что смотреть при оценке на газете

Чеклист для ручной проверки:

- [ ] Текстовые колонки выделены отдельными `text`-блоками
- [ ] Заголовки статей помечены как `paragraph_title` / `doc_title`
- [ ] Фото/иллюстрации попали в `image` / `figure`
- [ ] Порядок в `parsing_res_list` читается сверху вниз, колонка за колонкой
- [ ] Разные статьи на одной полосе не смешиваются (region detection)
- [ ] JSON/Markdown пригодны для дальнейшей разметки OCR-датасета

**Ограничение:** стандартная OCR-модель PaddleOCR рассчитана на китайский/английский текст.  
Для аварского языка качество **распознавания** будет низким, но **layout** и **reading order** можно оценивать уже сейчас.